In [ ]:
%%capture
%pip install qldpc
%pip install matplotlib

# Computing logical error rates

Here we show how to compute the logical error rates of error-correcting codes.  We consider both a code capacity model, in which a code accumulates errors that are corrected noiselessly, and a simulation of logical error rates in a quantum memory experiment with circuit-level noise.

In [ ]:
import os
from collections.abc import Sequence

import matplotlib.pyplot as plt
import numpy as np
import sinter

from qldpc import circuits, codes
from qldpc.objects import Pauli, PauliXZ

%matplotlib inline

## Code capacity

In the simplest code capacity model, physical errors are sampled by flipping each bit (or depolarizing a qubit) of the code with some probability.  The logical error rate is the fraction of physical errors that are decoded incorrectly, resulting in a logical error upon correction.  For a fixed noise model, this logical error rate is a joint property of (a) the code and (b) the decoder.

We start by writing a function to produce a plot of physical vs. logical error rates.

In [ ]:
def make_code_capacity_figure(
    codes_to_plot: Sequence[codes.ClassicalCode | codes.CSSCode],
    num_samples: int = 10**4,
    error_rates: Sequence[float] = list(np.logspace(-2, -0.1, 100)),
    distance_estimation_trials: bool | int = False,
    figsize: tuple[int, int] = (5, 4),
    **decoding_args: object,
) -> tuple[plt.Figure, plt.Axes]:
    """Plot physical vs. logical error rates for the given codes in a code capacity model.

    Args:
        codes_to_plot: the codes whose error rates we want to plot.
        num_samples: the number of times we sample physical errors.
        error_rates: the i.i.d. probabilities of physical errors on each bit/qubit.
        distance_estimation_trials: if the code distance is not know, estimate it with this many trials.
        figsize: the size of the figure to produce.
        **decoding_args: arguments to pass to the decoder.

    Returns:
        The matplotlib.pyplot figure and axis.
    """
    figure, axis = plt.subplots(figsize=figsize)

    for code in codes_to_plot:
        get_logical_error_rate = code.get_logical_error_rate_func(
            num_samples, max(error_rates), **decoding_args
        )
        logical_rates, stderrs = get_logical_error_rate(error_rates)
        label = get_label(code, distance_estimation_trials)
        line, *_ = axis.plot(error_rates, logical_rates, label=label)
        axis.fill_between(
            error_rates,
            logical_rates - stderrs,
            logical_rates + stderrs,
            color=line.get_color(),
            alpha=0.2,
        )

    axis.axline(
        (0, 0),
        slope=1,
        color="k",
        linestyle=":",
        label=r"$p_{\mathrm{log}}=p_{\mathrm{phys}}$",
    )
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xlim(right=1)
    axis.set_ylim(bottom=max(min(error_rates) ** 2, axis.get_ylim()[0]), top=1)
    axis.set_xlabel(r"physical error rate $p_{\mathrm{phys}}$")
    axis.set_ylabel(r"logical error rate $p_{\mathrm{log}}$")
    axis.legend(loc="best")
    axis.grid(which="both")
    figure.tight_layout()

    return figure, axis


def get_label(
    code: codes.ClassicalCode | codes.QuditCode,
    distance_estimation_trials: bool | int = False,
) -> str:
    """Get a label for a code in a figure."""
    known_distance = code.get_distance_if_known()
    if isinstance(known_distance, int):
        return f"$d={known_distance}$"
    if not distance_estimation_trials:
        return f"[{len(code)}, {code.dimension}]"
    distance_estimate = code.get_distance_bound(num_trials=int(distance_estimation_trials))
    return f"[{len(code)}, {code.dimension}, <= {distance_estimate}]"

### The repetition and surface codes

For these codes, we decode with minimum-weight perfect matching (MWPM).

In [ ]:
rep_codes = [codes.RepetitionCode(dist) for dist in [3, 5, 7]]
make_code_capacity_figure(rep_codes, with_MWPM=True)
plt.show()

In [ ]:
surface_codes = [codes.SurfaceCode(dist) for dist in [3, 5, 7]]
make_code_capacity_figure(surface_codes, with_MWPM=True)
plt.show()

### Bivariate bicycle codes

As introduced in [arXiv:2308.07915](https://arxiv.org/abs/2308.07915), these codes can be decoded with BD-OSD.

In [ ]:
from sympy.abc import x, y

bb_codes = [
    codes.BBCode(
        {x: 6, y: 6},
        x**3 + y + y**2,
        y**3 + x + x**2,
    ),
    codes.BBCode(
        {x: 15, y: 3},
        x**9 + y + y**2,
        1 + x**2 + x**7,
    ),
    codes.BBCode(
        {x: 9, y: 6},
        x**3 + y + y**2,
        y**3 + x + x**2,
    ),
]
make_code_capacity_figure(bb_codes, with_BP_OSD=True, distance_estimation_trials=100)
plt.show()

## Quantum memory with circuit-level noise

We now consider using a code as a quantum memory, and consider running one (noisy) logical QEC cycle.  The logical error rate is now the probability which which one logical QEC cycle induces a logical error.  For a fixed noise model, this logical error rate is a joint property of (a) the code, (b) the decoder, and (c) the syndrome measurement strategy.

More specifically, we consider small-scale memory experiments for the toric code:

- Build a toric code with code distance `d`.
- Compile syndrome-measurement circuits using an edge-coloring schedule.
- Inject i.i.d. depolarizing noise of strength `p`.
- Decode using BP+LSD + `sinter`.
- Sweep over `(d, p)` and estimate logical error rates for both Z- and X-type logical operators.

What is a “memory experiment”?  Prepare a logical state (e.g., `|0⟩_L` or `|+⟩_L`), perform `d` rounds of stabilizer measurements, then check if a logical operator has flipped at the end.  The resulting logical error rate as a function of the physical error rate lets you determine code the threshold and pseudothreshold of a code.

In [ ]:
def run_memory_experiments(basis: PauliXZ, distances: Sequence[int], error_rates: Sequence[float]):
    """Use sinter to simulate memory cycles of the toric code.

    This function ...
      1. Creates a toric code of each distance `d`.
      2. Builds a logical QEC cycle circuit using the `EdgeColoring` syndrome measurement strategy.
      3. Adds a depolarizing noise model with probability `p` to gates.
      4. Wraps each configuration as a `sinter.Task` with metadata `{d, p}`.
      5. Runs batched Monte Carlo sampling with `sinter.collect` using a BP+LSD decoder.

    Args:
        basis: the type of logical operator whose errors are tracked.
        distances: code distances to sweep for the toric code.
        error_rates: probabilities of error to sweep in a depolarizing noise model.

    Returns:
        A `sinter` stats object aggregating shots, errors, and metadata for downstream plotting.
    """
    syndrome_measurement_strategy = circuits.EdgeColoring()

    tasks: list[sinter.Task] = []
    noise_models = {
        prob: circuits.DepolarizingNoiseModel(prob, include_idling_error=False)
        for prob in error_rates
    }
    for distance in distances:
        code = codes.ToricCode(distance, rotated=False)
        circuit = circuits.get_memory_experiment(
            code, syndrome_measurement_strategy, num_rounds=distance, basis=basis
        )
        for prob in error_rates:
            noisy_circuit = noise_models[prob].noisy_circuit(circuit)
            tasks.append(
                sinter.Task(circuit=noisy_circuit, json_metadata={"d": distance, "p": prob})
            )

    return sinter.collect(
        num_workers=os.cpu_count() - 2,
        max_shots=10**5,
        max_errors=100,
        tasks=tasks,
        decoders=["bplsd"],
        custom_decoders={
            "bplsd": circuits.SinterDecoder(
                with_BP_LSD=True,
                max_iter=30,
                bp_method="ms",
                lsd_method="lsd_cs",
                lsd_order=0,
            )
        },
    )

In [ ]:
# Run the actual experiments.  Note that this might take a while...
distances = [3, 5, 7]
error_rates = np.logspace(-3, -2, 5)
z_basis_results = run_memory_experiments(Pauli.Z, distances, error_rates)
x_basis_results = run_memory_experiments(Pauli.X, distances, error_rates)

In [ ]:
fig, axes = plt.subplots(1, 2, sharey=True, figsize=(8, 4))

for axis, results, basis in zip(axes, [z_basis_results, x_basis_results], ["Z", "X"]):
    sinter.plot_error_rate(
        ax=axis,
        stats=z_basis_results,
        x_func=lambda stats: stats.json_metadata["p"],
        group_func=lambda stats: stats.json_metadata["d"],
    )
    axis.set_title(f"{basis} Basis")

    axis.loglog()
    axis.grid(which="both")
    axis.legend()
    axis.set_xlabel("Physical Error Rate")

plt.suptitle("Toric Code Memory Experiments", fontsize=14)
plt.show()